In [ ]:
from collections import namedtuple
from typing import Tuple

import os
import subprocess
import pickle
import numpy as np
import matplotlib.pyplot as plt

#### Utilities

In [ ]:
ColorsCounts = namedtuple("ColorsCounts", ["boyerMyrvold", "mst", "cactus"])
Times = namedtuple("Times", ["boyerMyrvold", "mst", "cactus"])

AverageResult = namedtuple(
    "AverageResult",
    ["basic", "weighted"]
)

TestResult = namedtuple(
    "TestResult", 
    ["colors", "times"]
)

In [ ]:
def test_minimal_coloring(n0: int, n: int, m: int, reps: int) -> Tuple[TestResult, TestResult] | None:
    try:
        result = subprocess.run(
            ["../build/minimum_coloring", str(5), str(n0), str(m), str(n), str(reps)],
            check=True,
            capture_output=True,
            text=True,
        )
    except subprocess.CalledProcessError as e:
        print(f"Error generating graph: {e}")
        return None
    
    output_lines = result.stdout.strip().splitlines()
    if len(output_lines) < 2:
        print("Unexpected output format")
        return None

    color_values = list(map(float, output_lines[0].split()))
    color_mean = AverageResult(
        basic=ColorsCounts(color_values[0], color_values[4], color_values[8]),
        weighted=ColorsCounts(color_values[2], color_values[6], color_values[10])
    )
    color_std = AverageResult(
        basic=ColorsCounts(color_values[1], color_values[5], color_values[9]),
        weighted=ColorsCounts(color_values[3], color_values[7], color_values[11])
    )

    time_values = list(map(float, output_lines[1].split()))
    time_mean = AverageResult(
        basic=Times(time_values[0], time_values[4], time_values[8]),
        weighted=Times(time_values[2], time_values[6], time_values[10])
    )
    time_std = AverageResult(
        basic=Times(time_values[1], time_values[5], time_values[9]),
        weighted=Times(time_values[3], time_values[7], time_values[11])
    )
    

    return TestResult(color_mean, time_mean), TestResult(color_std, time_std)

#### Finding Ranges For Coloring With Given Number of Colors